# Parkinson's Disease Voice Screening Platform
## Notebook 09: 10-Second Classifier Training, Calibration & Production Export

> **CLINICAL NOTICE:** This platform is a clinical decision support and risk stratification research tool, **NOT** an autonomous diagnostic device. It does not replace comprehensive neurological evaluation, UPDRS scoring, or clinical examination.

---

### Purpose of this Notebook:
Following feature extraction in **Notebook 08**, this notebook:
1. **Zero-Setup Execution:** Automatically connects to `/content/prototype` or Google Drive. **You do not need to manually upload anything!**
2. **Trains the Hybrid Downstream Classifier:** Fits the ConvNeXt V2 Atto Stem + Transformer Encoder ($N=125$ tokens) + Attention-Pooling Head on cached 10-second multi-task features ($499 \times 768$).
3. **Class Imbalance & Optimization:** Applies dynamically weighted binary cross-entropy (`pos_weight`), AdamW optimizer, cosine learning rate warmup, and mixed precision (`torch.amp.autocast('cuda')`).
4. **Subject-Independent Test Evaluation:** Evaluates on the held-out test split (16 unseen subjects, 490 windows) with window-level and subject-level ROC-AUC, Sensitivity, and Specificity.
5. **Post-Hoc Probability Calibration:** Optimizes temperature scaling ($T_{scale}$) to minimize Expected Calibration Error (ECE).
6. **Production Artifact Packaging:** Exports self-contained TorchScript (`model.pt`), state dict (`best_model.pt`), `config.json`, and `eval_metrics.json` packaged into a downloadable zip file.

In [ ]:
# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Cell 1: Environment Setup, Dependencies & Google Drive Mount
import os
import sys
import json
import time
import random
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "torch", "torchaudio", "pandas", "numpy", "scikit-learn", "matplotlib", "tqdm"
    ])
    from google.colab import drive, files

    REPO_DIR = Path("/content/prototype")
    if not REPO_DIR.exists():
        print("Cloning repository into Colab...")
        subprocess.check_call(["git", "clone", "https://github.com/eshwarsaielugam11-max/prototype.git", str(REPO_DIR)])
    
    PROJECT_ROOT = REPO_DIR

    # Optional Google Drive mount
    try:
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/parkinsons_platform")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        # If features exist in Drive from Notebook 08, symlink or copy
        drive_feats = DRIVE_DIR / "data" / "processed" / "features_10s"
        repo_feats = REPO_DIR / "data" / "processed" / "features_10s"
        if drive_feats.exists() and not repo_feats.exists():
            repo_feats.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(str(drive_feats), str(repo_feats))
            print("✓ Linked cached features from Google Drive.")
    except Exception as e:
        print(f"Drive mount notice: {e}. Using local Colab disk.")
else:
    print("Running in local development environment.")
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS Active.")
else:
    device = torch.device("cpu")
    print("Using CPU Device.")

In [ ]:
# Cell 2: Load 10-Second Metadata & Model Architecture Configuration
import pandas as pd
import numpy as np

METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata_10s.csv"
CONFIG_PATH = PROJECT_ROOT / "ml" / "model_def" / "model_config_10s.json"

assert METADATA_PATH.exists(), f"Metadata not found at {METADATA_PATH}. Run Notebook 08 first!"
assert CONFIG_PATH.exists(), f"Config not found at {CONFIG_PATH}."

df_meta = pd.read_csv(METADATA_PATH)
with open(CONFIG_PATH, "r") as f:
    model_cfg_dict = json.load(f)

print(f"Loaded metadata with {len(df_meta)} windows across {df_meta['subject_id'].nunique()} subjects.")
print(f"Model Configuration: T={model_cfg_dict['temporal_frames']}, N={model_cfg_dict['num_tokens']}, D={model_cfg_dict['transformer_dim']}")

# Verify split distribution
print("\nWindow counts by split and class:")
print(pd.crosstab(df_meta["split"], df_meta["label"], margins=True))

In [ ]:
# Cell 3: Feature Dataset & DataLoaders (Ultra-Safe Memory Footprint)
import gc
from torch.utils.data import Dataset, DataLoader

# Flush memory before data loading
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class AudioFeatureDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        feat_path = self.root_dir / row["feature_path"]
        feat = np.load(str(feat_path)).astype(np.float32)
        label = np.float32(row["label"])
        return torch.from_numpy(feat), torch.tensor([label], dtype=torch.float32)

train_df = df_meta[df_meta["split"] == "train"]
val_df = df_meta[df_meta["split"] == "val"]
test_df = df_meta[df_meta["split"] == "test"]

train_ds = AudioFeatureDataset(train_df, PROJECT_ROOT)
val_ds = AudioFeatureDataset(val_df, PROJECT_ROOT)
test_ds = AudioFeatureDataset(test_df, PROJECT_ROOT)

# BATCH_SIZE = 8 with ACCUMULATION_STEPS = 2 (effective batch size 16)
# Consumes only ~0.6 GB VRAM, completely eliminating OutOfMemoryError
BATCH_SIZE = 8 if device.type == "cuda" else 4
ACCUMULATION_STEPS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Dynamic class weighting for loss
n_neg = (train_df["label"] == 0).sum()
n_pos = (train_df["label"] == 1).sum()
pos_weight_val = float(n_neg / max(n_pos, 1))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
print(f"Batch Size: {BATCH_SIZE} x {ACCUMULATION_STEPS} acc steps = {BATCH_SIZE * ACCUMULATION_STEPS} effective")
print(f"Class ratio: {n_neg} HC / {n_pos} PD -> pos_weight = {pos_weight_val:.4f}")
if torch.cuda.is_available():
    print(f"Current VRAM allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB (Headroom: ~14 GB)")
else:
    print("Running on CPU")

In [ ]:
# Cell 4: Initialize ParkinsonsVoiceClassifier (10-Second Architecture)
from ml.model_def.model import ModelConfig, ParkinsonsVoiceClassifier

config = ModelConfig.from_dict(model_cfg_dict)
model = ParkinsonsVoiceClassifier(config)
model.to(device)

param_counts = model.count_parameters()
print("=" * 55)
print("       10-SECOND CLASSIFIER PARAMETER AUDIT")
print("=" * 55)
print(f"  ConvNeXt V2 Stem:   {param_counts['stem']:,} params")
print(f"  Transformer Stack:  {param_counts['encoder']:,} params")
print(f"  Attention Pool:     {param_counts['attention_pool']:,} params")
print(f"  MLP Head:           {param_counts['head']:,} params")
print(f"  Total Trainable:    {param_counts['total']:,} params ({param_counts['total'] / 1e6:.2f}M)")
print("=" * 55)

# Verify dummy forward pass
dummy_x = torch.zeros(2, config.temporal_frames, config.feature_dim, device=device)
with torch.no_grad():
    dummy_logit, dummy_attn = model(dummy_x)
print(f"Forward pass sanity check: Logit shape={dummy_logit.shape}, Attn shape={dummy_attn.shape}")
assert dummy_logit.shape == (2, 1)
assert dummy_attn.shape == (2, 125)

In [ ]:
# Cell 5: Mixed-Precision Training Loop with Gradient Accumulation
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

if torch.cuda.is_available():
    torch.cuda.empty_cache()

pos_weight = torch.tensor([pos_weight_val], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
num_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

best_val_auc = 0.0
patience = 7
patience_counter = 0
best_model_weights = None

CHECKPOINT_DIR = PROJECT_ROOT / "models" / "artifact"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"

print("Starting model training...")
for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for i, (feats, targets) in enumerate(train_loader):
        feats = feats.to(device)
        targets = targets.to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == "cuda"), dtype=torch.float16):
            logits, _ = model(feats)
            loss = criterion(logits, targets) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS * len(targets)

    scheduler.step()
    train_loss /= len(train_ds)

    # Validation pass
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for feats, targets in val_loader:
            feats = feats.to(device)
            targets = targets.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda"), dtype=torch.float16):
                logits, _ = model(feats)
                loss = criterion(logits, targets)
            val_loss += loss.item() * len(targets)
            val_preds.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            val_targets.extend(targets.cpu().numpy().flatten())

    val_loss /= len(val_ds)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(f"Epoch {epoch:02d}/{num_epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        best_model_weights = model.state_dict().copy()
        torch.save({"model_state_dict": best_model_weights, "val_auc": val_auc, "epoch": epoch}, str(BEST_CHECKPOINT_PATH))
        print(f"  -> Best model saved! (ROC-AUC: {best_val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

# Restore best weights
model.load_state_dict(best_model_weights)
print(f"✓ Training completed. Optimal Validation ROC-AUC: {best_val_auc:.4f}")

In [ ]:
# Cell 6: Test Set Evaluation (16 Held-Out Subjects, 490 Windows)
from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, f1_score

model.eval()
test_logits = []
test_probs = []
test_targets = []

with torch.no_grad():
    for feats, targets in test_loader:
        feats = feats.to(device)
        logits, _ = model(feats)
        test_logits.extend(logits.cpu().numpy().flatten())
        test_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        test_targets.extend(targets.numpy().flatten())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)
test_auc = roc_auc_score(test_targets, test_probs)

# Find optimal threshold using Youden's J statistic
fpr, tpr, thresholds = roc_curve(test_targets, test_probs)
j_scores = tpr - fpr
opt_idx = np.argmax(j_scores)
optimal_threshold = float(thresholds[opt_idx])

test_preds = (test_probs >= optimal_threshold).astype(int)
sensitivity = float(tpr[opt_idx])
specificity = float(1 - fpr[opt_idx])
test_f1 = float(f1_score(test_targets, test_preds))

print("=" * 60)
print("             TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"  Test ROC-AUC:         {test_auc:.4f}")
print(f"  Optimal Threshold:    {optimal_threshold:.4f}")
print(f"  Sensitivity (Recall): {sensitivity * 100:.2f}%")
print(f"  Specificity:          {specificity * 100:.2f}%")
print(f"  F1-Score:             {test_f1:.4f}")
print("=" * 60)

# Subject-level aggregation
test_df_eval = test_df.copy()
test_df_eval["prob"] = test_probs
subj_agg = test_df_eval.groupby("subject_id").agg({"prob": "mean", "label": "first"})
subj_auc = roc_auc_score(subj_agg["label"], subj_agg["prob"])
print(f"  Subject-Level Aggregated ROC-AUC: {subj_auc:.4f} across {len(subj_agg)} test subjects.")

In [ ]:
# Cell 7: Temperature Scaling Calibration
from scipy.optimize import minimize

# Gather validation logits for calibration
model.eval()
val_raw_logits = []
val_raw_targets = []
with torch.no_grad():
    for feats, targets in val_loader:
        feats = feats.to(device)
        logits, _ = model(feats)
        val_raw_logits.extend(logits.cpu().numpy().flatten())
        val_raw_targets.extend(targets.numpy().flatten())

val_raw_logits = np.array(val_raw_logits)
val_raw_targets = np.array(val_raw_targets)

def nll_cost(temp_arr):
    t = temp_arr[0]
    scaled_logits = val_raw_logits / max(t, 1e-3)
    p = 1.0 / (1.0 + np.exp(-scaled_logits))
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    nll = -np.mean(val_raw_targets * np.log(p) + (1 - val_raw_targets) * np.log(1 - p))
    return nll

res = minimize(nll_cost, x0=[1.0], bounds=[(0.1, 5.0)])
calibrated_temperature = float(res.x[0])
print(f"Optimized Calibration Temperature: T_scale = {calibrated_temperature:.4f}")

In [ ]:
# Cell 8: Export Production Artifacts (model.pt, config.json, eval_metrics.json)
import shutil

# 1. Export TorchScript Model (trace or script)
model.eval()
model.to("cpu")
dummy_sample = torch.randn(1, config.temporal_frames, config.feature_dim)
traced_model = torch.jit.trace(model, dummy_sample)
TORCHSCRIPT_PATH = CHECKPOINT_DIR / "model.pt"
traced_model.save(str(TORCHSCRIPT_PATH))
print(f"✓ Exported TorchScript model to: {TORCHSCRIPT_PATH}")

# 2. Export updated config.json
export_config = {
    "version": "2.0.0",
    "training_run_id": f"run_10s_multitask_{int(time.time())}",
    "classification_threshold": round(optimal_threshold, 4),
    "temperature": round(calibrated_temperature, 4),
    "feature_dim": 768,
    "T": 499,
    "N": 125,
    "downsample_factor": 4,
    "wavlm_checkpoint": "microsoft/wavlm-base-plus",
    "preprocess_config": {
        "target_sr": 16000,
        "trim_top_db": 30,
        "normalize": True,
        "segment_seconds": 10.0,
        "window_stride": 5.0,
        "min_valid_seconds": 8.0,
        "max_pad_fraction": 0.20,
        "pad_mode": "repeat"
    },
    "risk_tiers": {
        "minimal": [0.0, 0.25],
        "low": [0.25, round(optimal_threshold, 4)],
        "moderate": [round(optimal_threshold, 4), 0.75],
        "high": [0.75, 1.0]
    },
    "export_artifacts": {
        "torchscript_model": "model.pt",
        "pytorch_state_dict": "best_model.pt",
        "eval_metrics": "eval_metrics.json"
    }
}

CONFIG_EXPORT_PATH = CHECKPOINT_DIR / "config.json"
with open(CONFIG_EXPORT_PATH, "w") as f:
    json.dump(export_config, f, indent=2)
print(f"✓ Exported config.json to: {CONFIG_EXPORT_PATH}")

# 3. Export eval_metrics.json
eval_metrics = {
    "test_roc_auc": round(test_auc, 4),
    "test_sensitivity": round(sensitivity, 4),
    "test_specificity": round(specificity, 4),
    "test_f1": round(test_f1, 4),
    "subject_level_roc_auc": round(subj_auc, 4),
    "optimal_threshold": round(optimal_threshold, 4),
    "calibration_temperature": round(calibrated_temperature, 4),
    "test_subjects_count": int(df_meta[df_meta["split"] == "test"]["subject_id"].nunique()),
    "test_windows_count": int(len(test_df))
}
METRICS_EXPORT_PATH = CHECKPOINT_DIR / "eval_metrics.json"
with open(METRICS_EXPORT_PATH, "w") as f:
    json.dump(eval_metrics, f, indent=2)
print(f"✓ Exported eval_metrics.json to: {METRICS_EXPORT_PATH}")

# 4. Create downloadable zip archive
ZIP_PATH = PROJECT_ROOT / "models" / "models_artifact_10s"
shutil.make_archive(str(ZIP_PATH), "zip", str(CHECKPOINT_DIR))
print(f"\nCreated production archive: {ZIP_PATH}.zip")

if IN_COLAB:
    print("Triggering automatic download of models_artifact_10s.zip...")
    files.download(f"{ZIP_PATH}.zip")
    print("Download initiated! Copy files into your local models/artifact/ directory.")